<a href="https://colab.research.google.com/github/ernestotezza/CP_credit_rural_deforestation/blob/main/Scripts/1_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Run this cell with a Python runtime only.
# Switch back to R before running any cell below.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
setwd("/content/drive/MyDrive/CP_credit_rural_deforestation")
getwd()



[1] "/content/drive/MyDrive/CP_credit_rural_deforestation"

In [2]:
#load libraries
library(readr)
library(dplyr)
library(tidyr)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [3]:
#read in raw data

df_raw_credit_operations <- read_delim("raw_data/SICOR_OPERACAO_BASICA_ESTADO_2025.csv", delim=";")

Rows: 2456985 Columns: 47
── Column specification ────────────────────────────────────────────────────────
Delimiter: ";"
chr (16): CNPJ_IF, DT_EMISSAO, DT_VENCIMENTO, CD_FONTE_RECURSO, CNPJ_AGENTE_...
dbl (31): #REF_BACEN, NU_ORDEM, CD_INST_CREDITO, CD_CATEG_EMITENTE, CD_REF_B...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
head(df_raw_credit_operations)

colnames(df_raw_credit_operations)

#REF_BACEN,NU_ORDEM,CNPJ_IF,DT_EMISSAO,DT_VENCIMENTO,CD_INST_CREDITO,CD_CATEG_EMITENTE,CD_FONTE_RECURSO,CNPJ_AGENTE_INVEST,CD_ESTADO,⋯,DT_INIC_COLHEITA,DT_INIC_PLANTIO,VL_JUROS_ENC_FINAN_POSFIX,VL_PERC_CUSTO_EFET_TOTAL,CD_CONTRATO_STN,CD_CNPJ_CADASTRANTE,VL_AREA_INFORMADA,CD_CICLO_CULTIVAR,CD_TIPO_SOLO,PC_BONUS_CAR
<dbl>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
519338131,1,78483310,02/01/2025,17/01/2033,5,2222,0403,NA,SC,⋯,NA,NA,NA,6.08,2.024756e+12,02038232,0.03,NA,NA,NA
519338131,2,78483310,02/01/2025,17/01/2033,5,2222,0403,NA,SC,⋯,NA,NA,NA,6.08,2.024756e+12,02038232,0.06,NA,NA,NA
519338132,1,87733770,02/01/2025,15/11/2034,5,2222,0403,NA,RS,⋯,NA,NA,NA,NA,2.024748e+12,01181521,2.06,NA,NA,NA
519338133,1,78825023,02/01/2025,04/08/2025,5,2222,0403,NA,RS,⋯,05/05/2025,15/01/2025,NA,3.68,2.024756e+12,02038232,1.00,NA,NA,NA
519338134,1,78825023,02/01/2025,13/08/2025,5,2222,0403,NA,RS,⋯,15/05/2025,25/01/2025,NA,3.66,2.024756e+12,02038232,2.93,NA,NA,NA
519338135,1,78825023,02/01/2025,11/08/2025,5,2222,0201,NA,RS,⋯,11/05/2025,03/02/2025,NA,3.66,NA,02038232,1.80,NA,NA,NA


[1] "#REF_BACEN"                "NU_ORDEM"                 
 [3] "CNPJ_IF"                   "DT_EMISSAO"               
 [5] "DT_VENCIMENTO"             "CD_INST_CREDITO"          
 [7] "CD_CATEG_EMITENTE"         "CD_FONTE_RECURSO"         
 [9] "CNPJ_AGENTE_INVEST"        "CD_ESTADO"                
[11] "CD_REF_BACEN_INVESTIMENTO" "CD_TIPO_SEGURO"           
[13] "CD_EMPREENDIMENTO"         "CD_PROGRAMA"              
[15] "CD_TIPO_ENCARG_FINANC"     "CD_TIPO_IRRIGACAO"        
[17] "CD_TIPO_AGRICULTURA"       "CD_FASE_CICLO_PRODUCAO"   
[19] "CD_TIPO_CULTIVO"           "CD_TIPO_INTGR_CONSOR"     
[21] "CD_TIPO_GRAO_SEMENTE"      "VL_ALIQ_PROAGRO"          
[23] "VL_JUROS"                  "VL_PRESTACAO_INVESTIMENTO"
[25] "VL_PREV_PROD"              "VL_QUANTIDADE"            
[27] "VL_RECEITA_BRUTA_ESPERADA" "VL_PARC_CREDITO"          
[29] "VL_REC_PROPRIO"            "VL_PERC_RISCO_STN"        
[31] "VL_PERC_RISCO_FUNDO_CONST" "VL_REC_PROPRIO_SRV"       
[33] "VL_AREA_FINANC"            "CD_SUBPROGRAMA"           
[35] "VL_PRODUTIV_OBTIDA"        "DT_FIM_COLHEITA"          
[37] "DT_FIM_PLANTIO"            "DT_INIC_COLHEITA"         
[39] "DT_INIC_PLANTIO"           "VL_JUROS_ENC_FINAN_POSFIX"
[41] "VL_PERC_CUSTO_EFET_TOTAL"  "CD_CONTRATO_STN"          
[43] "CD_CNPJ_CADASTRANTE"       "VL_AREA_INFORMADA"        
[45] "CD_CICLO_CULTIVAR"         "CD_TIPO_SOLO"             
[47] "PC_BONUS_CAR"

In [13]:
#cleaning datasets

#simple cleaning now, this will improve

df_clean_credit_operations <- df_raw_credit_operations %>%
  group_by(CD_ESTADO) %>%
  summarise(
      total_contracted_credit_amout = sum(VL_PARC_CREDITO)
    ) %>%
  arrange(desc(total_contracted_credit_amout)) %>%
  mutate(value_in_reais = paste0("R$ ", formatC(
    total_contracted_credit_amout,
    format = "f", digits = 2,
    big.mark = ".", decimal.mark = ","
  )))

df_clean_credit_operations

CD_ESTADO,total_contracted_credit_amout,value_in_reais
<chr>,<dbl>,<chr>
MG,48310249291,"R$ 48.310.249.290,92"
PR,46876710674,"R$ 46.876.710.673,69"
RS,44830429446,"R$ 44.830.429.446,09"
MT,38347686522,"R$ 38.347.686.522,18"
SP,37597085120,"R$ 37.597.085.120,45"
GO,29207171425,"R$ 29.207.171.425,41"
SC,22591155751,"R$ 22.591.155.750,77"
MS,15921144185,"R$ 15.921.144.184,67"
BA,14982807742,"R$ 14.982.807.741,82"
